In [3]:
import pandas as pd
import numpy as np
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
# ===========================================================
# zkp_bond_simulation_FINAL.py — BUG FIXED
# Fix: adoption prob uses income_decile (0-9), not raw INCWAGE
# ===========================================================
 
INPUT_FILE   = 'ZKP_TARGET_HOUSEHOLDS.csv'
OUTPUT_FILE  = 'ZKP_BOND_SIMULATION_RESULTS.csv'
PUMA_ZKP_OUT = 'ZKP_PUMA_IMPACT.csv'
ZKP_CAP      = 25000
YIELD_RATE   = 0.055
TAX_CREDIT   = 0.15
BOND_TERM    = 3
RISK_FREE    = 0.043
SCALE_FACTOR = 4.3
LAMBDA       = 0.18
BASE_ADOPT   = 0.12
PRIVACY_MULT = 1.45
 
# STEP 1: LOAD
print('Loading ZKP target households...')
df = pd.read_csv(INPUT_FILE, dtype={'GEOID_JOIN':str})
df['GEOID_JOIN'] = df['GEOID_JOIN'].astype(str).str.zfill(7)
print(f'  {len(df):,} households loaded')
 
# STEP 2: FIX ZKP_INVEST_POTENTIAL IF NEEDED
pct_zero = (df['ZKP_INVEST_POTENTIAL']==0).mean()
if pct_zero > 0.50:
    print(f'  Recomputing ZKP_INVEST_POTENTIAL ({pct_zero*100:.1f}% zero)...')
    df['TOTAL_ECONOMIC_CAPACITY'] = df['INCWAGE']+df['SHADOW_LIQUIDITY_GAP'].clip(lower=0)
    df['ZKP_INVEST_POTENTIAL'] = (df['TOTAL_ECONOMIC_CAPACITY']*0.20).clip(0,ZKP_CAP)
print(f'  ZKP_INVEST_POTENTIAL mean: ${df["ZKP_INVEST_POTENTIAL"].mean():,.0f}')
 
# STEP 3: NPV OF INCENTIVE
df['NPV_TAX_BENEFIT'] = TAX_CREDIT * df['ZKP_INVEST_POTENTIAL']
df['NPV_YIELD_3YR']   = sum(
    df['ZKP_INVEST_POTENTIAL']*YIELD_RATE/(1+RISK_FREE)**t for t in range(1,BOND_TERM+1))
df['NPV_TOTAL_INCENTIVE'] = df['NPV_TAX_BENEFIT'] + df['NPV_YIELD_3YR']
 
# STEP 4: ADOPTION PROBABILITY
df['ADOPTION_PROB_RAW'] = 1-np.exp(-LAMBDA*df['NPV_TOTAL_INCENTIVE']/10000)
df['ADOPTION_PROB'] = (
    BASE_ADOPT + (0.85-BASE_ADOPT)*df['ADOPTION_PROB_RAW']*PRIVACY_MULT
).clip(0,0.85)
df['ADOPTION_PROB'] = df['ADOPTION_PROB'] * np.where(df['RECENT_ARRIVAL']==1,0.80,1.0)
df['ADOPTION_PROB'] = df['ADOPTION_PROB'] * np.where(df['IS_CITIZEN']==0,0.90,1.0)
# FIX: use income DECILE (0-9), NOT raw INCWAGE (which causes overflow to ceiling)
incwage_decile = pd.qcut(df['INCWAGE'], q=10, labels=False, duplicates='drop').fillna(0)
df['ADOPTION_PROB'] = (df['ADOPTION_PROB']*(1+incwage_decile*0.015)).clip(0,0.85)
print(f'  ADOPTION_PROB: {df["ADOPTION_PROB"].min():.4f} to {df["ADOPTION_PROB"].max():.4f}')
print(f'  At ceiling (0.85): {(df["ADOPTION_PROB"]>=0.849).sum()} (should be near 0)')
 
# STEP 5: MONTE CARLO (1,000 iterations)
print('Running Monte Carlo (1,000 iterations)...')
np.random.seed(SEED := 42)
N_SIM, results = 1000, []
for _ in range(N_SIM):
    adopted  = np.random.binomial(1, df['ADOPTION_PROB'].values)
    invested = adopted * df['ZKP_INVEST_POTENTIAL'].values
    remit    = df['EST_ANNUAL_REMITTANCE_MODEL'].values
    ro       = np.where(remit>0, np.minimum(invested/remit,1.0), 0.0)
    results.append({'adopters':adopted.sum(),'capital_retained':invested.sum(),
                    'avg_invest_hh':invested[adopted==1].mean() if adopted.sum()>0 else 0,
                    'remit_offset_avg':ro[adopted==1].mean() if adopted.sum()>0 else 0})
sim_df = pd.DataFrame(results)
print()
print('-'*55)
print('MONTE CARLO RESULTS (n=1,000)')
print('-'*55)
ratio_cols = {'remit_offset_avg'}
for col in sim_df.columns:
 fmt = '.4f' if col in ratio_cols else ',.0f'
 print(f'{col:25s}: {sim_df[col].mean():>12{fmt}} +/- {sim_df[col].std():>10{fmt}}')

print('-'*55)
 
# STEP 6: DETERMINISTIC OUTPUT
np.random.seed(42)
df['ADOPTED_BOND']         = np.random.binomial(1,df['ADOPTION_PROB'].values)
df['CAPITAL_RETAINED']     = df['ADOPTED_BOND']*df['ZKP_INVEST_POTENTIAL']
df['REMITTANCE_PREVENTED'] = df['ADOPTED_BOND']*df['EST_ANNUAL_REMITTANCE_MODEL']
df['TAX_CREDIT_VALUE']     = df['ADOPTED_BOND']*df['ZKP_INVEST_POTENTIAL']*TAX_CREDIT
df['YIELD_INCOME_3YR']     = df['ADOPTED_BOND']*df['ZKP_INVEST_POTENTIAL']*YIELD_RATE*BOND_TERM
 
# STEP 7: SCALE-UP
retained_s = df['CAPITAL_RETAINED'].sum()
retained_n = retained_s * SCALE_FACTOR
theoretical = df['ZKP_INVEST_POTENTIAL'].sum()*0.35*SCALE_FACTOR
print()
print('-'*55)
print('CAPITAL RETENTION ESTIMATES')
print('-'*55)
print(f'Year-1 cold-start:')
print(f'  Adopters: {df["ADOPTED_BOND"].sum():,}  ({df["ADOPTED_BOND"].mean()*100:.1f}%)')
print(f'  Retained (sample): ${retained_s/1e6:.2f}M')
print(f'  Retained (national x{SCALE_FACTOR}): ${retained_n/1e6:.1f}M')
print(f'Mature program (35% take-up):')
print(f'  Retained (national): ${theoretical/1e6:.1f}M')
print(f'  ROI: {1/TAX_CREDIT:.1f}x')
print('-'*55)
 
# STEP 8: PUMA IMPACT TABLE
puma_zkp = df.groupby(['GEOID_JOIN','STATE_NAME','COUNTRY_NAME']).agg(
    hh_targeted=('ADOPTION_PROB','count'), hh_adopted=('ADOPTED_BOND','sum'),
    avg_adoption_prob=('ADOPTION_PROB','mean'), capital_retained=('CAPITAL_RETAINED','sum'),
    remittance_prevented=('REMITTANCE_PREVENTED','sum'),
    tax_credit_cost=('TAX_CREDIT_VALUE','sum'), yield_income=('YIELD_INCOME_3YR','sum'),
).reset_index().round(2)
puma_zkp['adoption_rate']    = (puma_zkp['hh_adopted']/puma_zkp['hh_targeted']).round(3)
puma_zkp['net_govt_benefit'] = puma_zkp['remittance_prevented']-puma_zkp['tax_credit_cost']
puma_zkp['roi_ratio']        = (puma_zkp['capital_retained']/puma_zkp['tax_credit_cost'].replace(0,1)).round(2)
puma_zkp = puma_zkp.sort_values('capital_retained',ascending=False)
puma_zkp['GEOID_JOIN'] = puma_zkp['GEOID_JOIN'].astype(str).str.zfill(7)
 
# STEP 9: CHARTS
country_data = df.groupby('COUNTRY_NAME').agg(
    retained=('CAPITAL_RETAINED','sum'),
    tax_cost=('TAX_CREDIT_VALUE','sum'),
    remit_prev=('REMITTANCE_PREVENTED','sum'),
).reset_index()
fig,axes = plt.subplots(1,2,figsize=(14,6))
x = range(len(country_data))
names = country_data['COUNTRY_NAME'].tolist()
axes[0].bar([i-0.2 for i in x],country_data['retained']/1e6,
            width=0.4,color='#0D2B4E',label='Capital Retained ($M)')
axes[0].bar([i+0.2 for i in x],country_data['tax_cost']/1e6,
            width=0.4,color='#B22222',label='Tax Credit Cost ($M)')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(names,rotation=35,ha='right',fontsize=10)
axes[0].set_title('ZKP Bond: Capital Retained vs. Tax Cost\nby Country of Origin',
                  fontsize=11,fontweight='bold')
axes[0].set_ylabel('USD (Millions)',fontsize=11)
axes[0].legend(fontsize=10)
country_data['roi'] = country_data['retained']/country_data['tax_cost'].replace(0,1)
bcols = ['#1A6B3C' if r>5 else '#0E6655' if r>3 else '#B8860B' for r in country_data['roi']]
bars = axes[1].bar(names,country_data['roi'],color=bcols)
axes[1].axhline(y=1,color='red',linestyle='--',label='Break-even (1:1)')
axes[1].set_title('ZKP Bond Return on Investment\n(Capital Retained per $1 Tax Credit)',
                  fontsize=11,fontweight='bold')
axes[1].set_ylabel('Capital Retained per $1 of Tax Credit',fontsize=11)
axes[1].tick_params(axis='x',rotation=35)
axes[1].legend(fontsize=10)
for bar,val in zip(bars,country_data['roi']):
    axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.05,
                 f'{val:.1f}x',ha='center',fontsize=10,fontweight='bold')
 
# STEP 10: SAVE
df['GEOID_JOIN'] = df['GEOID_JOIN'].astype(str).str.zfill(7)
df.to_csv(OUTPUT_FILE, index=False, quoting=csv.QUOTE_NONNUMERIC)
puma_zkp.to_csv(PUMA_ZKP_OUT, index=False, quoting=csv.QUOTE_NONNUMERIC)
plt.tight_layout()
plt.savefig('zkp_bond_cost_benefit.png',dpi=300,bbox_inches='tight')
plt.close()
print()
print('='*60)
print('ZKP SIMULATION COMPLETE')
print('='*60)
print(f'Results:   {OUTPUT_FILE}')
print(f'PUMA:      {PUMA_ZKP_OUT}')
print(f'Chart:     zkp_bond_cost_benefit.png')
print(f'Adopters:  {df["ADOPTED_BOND"].sum():,}  ({df["ADOPTED_BOND"].mean()*100:.1f}%)')
print(f'Retained:  ${df["CAPITAL_RETAINED"].sum()/1e6:.2f}M (sample)')
print(f'National:  ${retained_n/1e6:.1f}M (Year-1)  |  ${theoretical/1e6:.1f}M (mature)')
print(f'ROI:       {1/TAX_CREDIT:.1f}x')
print('ADOPTION_PROB check:', (df['ADOPTION_PROB']>=0.849).sum(), 'at ceiling (should be ~0)')
print('='*60)

Loading ZKP target households...
  24,092 households loaded
  ZKP_INVEST_POTENTIAL mean: $1,343
  ADOPTION_PROB: 0.0864 to 0.2186
  At ceiling (0.85): 0 (should be near 0)
Running Monte Carlo (1,000 iterations)...

-------------------------------------------------------
MONTE CARLO RESULTS (n=1,000)
-------------------------------------------------------
adopters                 :        2,727 +/-         49
capital_retained         :    4,001,091 +/-    108,749
avg_invest_hh            :        1,467 +/-         30
remit_offset_avg         :       0.2841 +/-     0.0058
-------------------------------------------------------

-------------------------------------------------------
CAPITAL RETENTION ESTIMATES
-------------------------------------------------------
Year-1 cold-start:
  Adopters: 2,760  (11.5%)
  Retained (sample): $4.07M
  Retained (national x4.3): $17.5M
Mature program (35% take-up):
  Retained (national): $48.7M
  ROI: 6.7x
---------------------------------------------